# C-02 LangChain + langchain-ollama

이 노트북은 LangChain의 `ChatOllama` 래퍼를 사용해 로컬 Ollama 모델을 호출하는 방식입니다.

## LangChain이 하는 일

LangChain은 LLM 호출을 애플리케이션 코드에서 다루기 쉽게 추상화하는 라이브러리입니다. 모델 공급자가 Ollama인지 OpenAI인지에 따라 API 모양이 조금씩 다른데, LangChain은 이를 비슷한 인터페이스로 감싸 줍니다.

## 이 노트북에서 보는 포인트

- `ChatOllama`로 로컬 Ollama 모델을 LangChain 채팅 모델처럼 사용합니다.
- Pydantic 모델을 `with_structured_output`에 넘겨 구조화된 결과를 받습니다.
- 직접 HTTP payload를 만들지 않고 `invoke()`로 실행합니다.

## 직접 호출 방식과 차이

C-01은 HTTP 요청을 직접 만들기 때문에 가장 투명합니다. C-02는 LangChain이 요청 형식과 구조화 출력 처리를 대신 다룹니다. 코드가 짧아지는 대신 LangChain의 동작 방식과 버전별 차이를 이해해야 합니다.

## 언제 적합한가

- 여러 LLM 공급자를 바꿔 가며 비교하고 싶을 때
- 프롬프트 템플릿, 체인, 구조화 출력 같은 LangChain 기능을 활용할 때
- 나중에 도구 호출이나 복수 단계 체인으로 확장할 가능성이 있을 때

참고: 이 예제는 LangChain 전체 기능 중 아주 작은 부분만 사용합니다. 핵심은 “Ollama 직접 호출보다 한 단계 추상화된 호출 방식”을 보는 것입니다.

In [ ]:
import json
import os
from typing import Literal

from pydantic import BaseModel, Field

try:
    from langchain_ollama import ChatOllama
except ImportError as exc:
    raise RuntimeError("이 노트북을 실행하려면 langchain-ollama를 설치하세요") from exc

# LangChain 래퍼를 써도 실제 모델은 로컬 Ollama에 설치되어 있어야 합니다.
MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:latest")

In [ ]:
class EmailAnalysisResult(BaseModel):
    # LangChain의 structured output은 이 Pydantic 스키마를 보고 모델 응답을 구조화합니다.
    # predicted_email_intent 라벨은 임시 업무 분류 체계입니다. 실제 운영 전 반드시 사용자 검토가 필요합니다.
    # - inquiry: 견적/납기/제품 문의처럼 아직 발주가 확정되지 않은 요청
    # - order: 구매 발주서, PO, 주문 확정처럼 실제 주문 처리로 이어지는 요청
    # - service: 클레임, 고장, 누수, 긴급 지원처럼 서비스/AS 대응이 필요한 요청
    # - technical: 도면, 사양, 기술 검토, 호환성 확인처럼 엔지니어링 판단이 필요한 요청
    # - other: 위 기준으로 분류하기 어렵거나 추가 업무 유형 정의가 필요한 메일
    # TODO: 실제 고객 메일 샘플을 보고 라벨 이름, 개수, 정의를 확정해야 합니다.
    predicted_email_intent: Literal["inquiry", "order", "service", "technical", "other"]
    # predicted_email_importance 라벨도 임시 우선순위 체계입니다. SLA, 업무 프로세스, 사용자 화면 정책에 맞게 조정해야 합니다.
    # - low: 참고/일반 정보성으로 즉시 처리가 필요하지 않은 메일
    # - normal: 통상 처리 기한 안에 대응하면 되는 일반 업무 메일
    # - high: 납기, 견적 마감, 고객 영향 등으로 우선 확인이 필요한 메일
    # - urgent: 긴급 수리, 선박 운항 영향, 즉시 회신 요구처럼 지연 시 손실이 큰 메일
    # TODO: predicted_email_importance는 단순 감정/단어가 아니라 실제 처리 SLA와 연결해 정의해야 합니다.
    predicted_email_importance: Literal["low", "normal", "high", "urgent"]
    # extracted_key_information은 모델이 이메일에서 추출한 핵심 업무 정보입니다.
    # 예: 견적번호, PO 번호, 제품명, 수량, 납기일, 선박명 등.
    # TODO: 실제 필드 목록이 확정되면 dict가 아니라 별도 Pydantic 모델로 바꾸는 것이 좋습니다.
    extracted_key_information: dict = Field(default_factory=dict)
    # predicted_assignee_area는 실제 개인 담당자라기보다 임시 담당 영역/팀 후보입니다.
    # 예: sales_team, service_team, technical_team, order_management 등.
    # TODO: 실제 사용자/팀/라우팅 규칙이 정리되면 assignee_user_id, assignee_team_id와 분리할지 결정해야 합니다.
    predicted_assignee_area: str
    # prediction_reasoning은 위 예측값을 낸 근거 설명입니다.
    # TODO: 실제 UI에 노출할지, 내부 감사/디버깅 용도로만 저장할지 결정해야 합니다.
    prediction_reasoning: str
    # predicted_needs_human_review는 AI가 사람 검토 필요성을 예측한 값입니다.
    # 최종 검토 상태가 아니며, 서비스 계층에서 정책/신뢰도/오류 여부와 함께 확정해야 합니다.
    predicted_needs_human_review: bool


# 서비스 요청 성격이 뚜렷한 한글 샘플입니다.
sample_email = {
    "subject": "밸브 누수 긴급 서비스 요청",
    "sender": "shipyard@example.com",
    "body": "검사 중 설치된 밸브에서 누수가 확인되었습니다. 교체 부품과 긴급 서비스 지원 가능 여부를 안내 부탁드립니다.",
    "attachments": ["누수_사진.jpg", "검사_보고서.pdf"],
}

In [ ]:
# ChatOllama는 LangChain 표준 ChatModel 인터페이스를 따릅니다.
llm = ChatOllama(model=MODEL, temperature=0)

# Pydantic 모델을 붙이면 응답을 EmailAnalysisResult 형태로 받으려 시도합니다.
# 직접 requests를 쓸 때보다 JSON 파싱 코드가 줄어드는 것이 장점입니다.
structured_llm = llm.with_structured_output(EmailAnalysisResult)

prompt = f"""
선박 부품 제조사 업무 이메일을 분석하세요.
스키마에서 요구하는 구조화된 결과만 반환하세요.

이메일:
{json.dumps(sample_email, ensure_ascii=False, indent=2)}
"""

result = structured_llm.invoke(prompt)
print(result.model_dump_json(indent=2))